# Interações, resíduos e qualidade do ajuste

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_07/03_interacoes_residuos_e_qualidade.ipynb)

Uma interação representa uma associação que varia conforme outra característica.
O diagnóstico, por sua vez, pergunta onde o modelo falha. Ambos impedem que uma
única média seja tomada como retrato uniforme do corpus.

![Duas retas com inclinações distintas representam uma interação.](imagens/03_interacao.svg)

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_07'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

## 1. Interação: quando a inclinação depende do contexto

Investigaremos se a associação entre urbanização e equipamentos varia com a
escolaridade. Centralizar as duas variáveis faz o intercepto e os termos principais
se referirem a valores médios observados.

$$Y_i = \beta_0 + \beta_1 U_{ci} + \beta_2 E_{ci}
+ \beta_3(U_{ci}E_{ci}) + \varepsilon_i$$

Com interação, $\beta_1$ não é uma inclinação universal: a inclinação de
urbanização é $\beta_1 + \beta_3 E_c$.

O código primeiro cria versões centralizadas e seu produto; depois usa a função de
ajuste já conhecida. Observe essa sequência conceitual, não a sintaxe algébrica.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

dados = pd.read_csv("dados/municipios_culturais.csv")
dados["urbanizacao_c"] = dados["urbanizacao_pct"] - dados["urbanizacao_pct"].mean()
dados["escolaridade_c"] = dados["escolaridade_media_anos"] - dados["escolaridade_media_anos"].mean()
dados["interacao"] = dados["urbanizacao_c"] * dados["escolaridade_c"]

In [ ]:
def ajustar_ols(X, y, nomes):
    X_com_intercepto = np.column_stack([np.ones(len(X)), np.asarray(X, dtype=float)])
    coeficientes = np.linalg.lstsq(X_com_intercepto, np.asarray(y, dtype=float), rcond=None)[0]
    previstos = X_com_intercepto @ coeficientes
    residuos = np.asarray(y, dtype=float) - previstos
    r2 = 1 - np.sum(residuos ** 2) / np.sum((y - np.mean(y)) ** 2)
    return {
        "X": X_com_intercepto, "coeficientes": coeficientes,
        "previstos": previstos, "residuos": residuos,
        "r2": r2, "rmse": np.sqrt(np.mean(residuos ** 2)),
        "mae": np.mean(np.abs(residuos)), "nomes": ["intercepto", *nomes],
    }

def tabela_coeficientes(modelo):
    return pd.DataFrame({"termo": modelo["nomes"], "coeficiente": modelo["coeficientes"]})

In [ ]:
nomes_interacao = ["urbanizacao_c", "escolaridade_c", "interacao"]
modelo_interacao = ajustar_ols(dados[nomes_interacao], dados["equipamentos_culturais_100mil"], nomes_interacao)
display(tabela_coeficientes(modelo_interacao).round(4))

grades = np.linspace(dados["urbanizacao_pct"].min(), dados["urbanizacao_pct"].max(), 80)
for escolaridade in [8, 10, 12]:
    uc = grades - dados["urbanizacao_pct"].mean()
    ec = escolaridade - dados["escolaridade_media_anos"].mean()
    matriz = np.column_stack([np.ones(len(grades)), uc, np.repeat(ec, len(grades)), uc * ec])
    plt.plot(grades, matriz @ modelo_interacao["coeficientes"], label=f"{escolaridade} anos")
plt.xlabel("Urbanização (%)"); plt.ylabel("Equipamentos previstos por 100 mil")
plt.title("Previsões condicionais do modelo de interação")
plt.legend(title="Escolaridade"); plt.tight_layout(); plt.show()

## 2. Diagnóstico: voltar dos resumos aos casos

Resíduos grandes identificam casos mal descritos; alavancagem alta identifica
combinações incomuns de explicativas. Nenhuma é defeito automático. São convites
para conferir fonte, codificação, contexto e dependência do resultado.

![O diagnóstico confronta previstos, resíduos, casos influentes e decisão analítica.](imagens/03_diagnostico.svg)

In [ ]:
X = modelo_interacao["X"]
alavancagem = np.diag(X @ np.linalg.pinv(X.T @ X) @ X.T)
diagnostico = dados[["id_municipio", "municipio", "equipamentos_culturais_100mil"]].copy()
diagnostico["previsto"] = modelo_interacao["previstos"]
diagnostico["residuo"] = modelo_interacao["residuos"]
diagnostico["alavancagem"] = alavancagem
display(diagnostico.reindex(diagnostico["residuo"].abs().sort_values(ascending=False).index).head(6).round(3))

fig, eixos = plt.subplots(1, 2, figsize=(11, 4))
eixos[0].scatter(modelo_interacao["previstos"], modelo_interacao["residuos"], alpha=.6)
eixos[0].axhline(0, color="#a44a3f"); eixos[0].set(xlabel="Previstos", ylabel="Resíduos", title="Resíduos × previstos")
eixos[1].scatter(dados["equipamentos_culturais_100mil"], modelo_interacao["previstos"], alpha=.6)
limites = [dados["equipamentos_culturais_100mil"].min(), dados["equipamentos_culturais_100mil"].max()]
eixos[1].plot(limites, limites, color="#a44a3f"); eixos[1].set(xlabel="Observados", ylabel="Previstos", title="Observados × previstos")
plt.tight_layout(); plt.show()
print(f"R²={modelo_interacao['r2']:.3f} | RMSE={modelo_interacao['rmse']:.3f} | MAE={modelo_interacao['mae']:.3f}")

## 3. Sensibilidade a um caso atípico

Remover um caso só para melhorar o resultado seria inadequado. Como o arquivo de
proveniência declara uma observação didaticamente alterada, fazemos uma análise de
sensibilidade: comparamos resultados com e sem ela e mantemos as duas versões.

In [ ]:
sem_m150 = dados[dados["id_municipio"] != "M150"]
modelo_sem_m150 = ajustar_ols(sem_m150[nomes_interacao], sem_m150["equipamentos_culturais_100mil"], nomes_interacao)
sensibilidade = pd.DataFrame({
    "modelo": ["todos os casos", "sem M150"],
    "R²": [modelo_interacao["r2"], modelo_sem_m150["r2"]],
    "RMSE": [modelo_interacao["rmse"], modelo_sem_m150["rmse"]],
    "coef_interacao": [modelo_interacao["coeficientes"][3], modelo_sem_m150["coeficientes"][3]],
})
display(sensibilidade.round(3))

### U07-A04 — Atividade — interação e diagnóstico

1. Explique por que o coeficiente de urbanização depende da escolaridade.
2. Leia as três linhas previstas sem comparar apenas o coeficiente de interação.
3. Inspecione o maior resíduo na base e na proveniência.
4. Compare o modelo completo e a sensibilidade; justifique qual relatar.

**Dinâmica (35 minutos):** duplas produzem uma figura comentada e um parágrafo de
cautela. Na plenária, outra dupla deve conseguir reconstruir a decisão tomada.

**Resposta:** escreva aqui.

## Passagem

A resposta foi contínua até agora. Para presença ou ausência, a reta pode prever
valores abaixo de zero ou acima de um. A regressão logística modela a ocorrência
respeitando o intervalo da probabilidade.